# Used OSS by Application and Environment Report

Generate comprehensive OSS usage reports from Contrast Security's `/libraries/filter` API.

**Outputs:**
- Markdown summary grouped by application and environment  
- CSV with flattened library/app/environment rows including library name, version, latest version, CVEs, usage, and SHA1 hash

**Features:**
- Automatic pagination through large result sets (offset/limit with fallback)
- Derives output filename prefix from environment section label
- Maps applications to environments via server associations
- Flattens one row per library per application per environment

## Notebook Flow

1. Run Code Cell 1 to set environment variables manually in Jupyter.
2. Optionally run Code Cell 2 to load the same variables from root `.env`.
3. Run Code Cell 3 to configure runtime and output paths.
4. Run Code Cell 4 to fetch data from Contrast APIs.
5. Run Code Cell 5 to generate CSV and Markdown outputs.
6. Run Code Cell 6 to view summary and output file locations.

### Notes

- Root credentials file is `[repo-root]/.env`.
- Example template is `[repo-root]/example.env`.
- Output for this notebook is in `notebooks/used_OSS_by_app/Output/`.
- Filename pattern: `{section}_used_oss_by_app_YYYY-MM-DD.{csv,md}`.

In [ ]:

# Cell 1 of 2: Set credentials manually
# Fill in your values here, then run this cell.
import os

os.environ["TEAMSERVER_URL"] = "https://your_saas_instance.contrastsecurity.com/"
os.environ["ORG_UUID"]       = ""
os.environ["CONTRAST_AUTH"]  = ""
os.environ["CONTRAST_API_KEY"] = ""


In [ ]:
# Or you can load these values from a .env file. Just make sure to fill in the values in the .env file first, then run this cell.
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

In [ ]:

# Runtime configuration — reads env vars set above
import os
from pathlib import Path
from datetime import datetime, timezone

def find_repo_root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p / "notebooks").exists() and (p / "README.md").exists():
            return p
    return Path(start).resolve()

def get_env(*names):
    for name in names:
        value = os.environ.get(name)
        if value:
            return value
    return None

teamserver_url   = get_env("TEAMSERVER_URL", "TeamserverURL", "url")
org_uuid         = get_env("ORG_UUID", "organizationId")
contrast_auth    = get_env("CONTRAST_AUTH", "AUTH", "authHeader")
contrast_api_key = get_env("CONTRAST_API_KEY", "API_KEY", "apiKey")
section_name     = "report"

_missing = []
if not teamserver_url:
    _missing.append("TEAMSERVER_URL (or TeamserverURL)")
if not org_uuid:
    _missing.append("ORG_UUID")
if not contrast_auth:
    _missing.append("CONTRAST_AUTH (or AUTH)")
if not contrast_api_key:
    _missing.append("CONTRAST_API_KEY (or API_KEY)")
if _missing:
    raise EnvironmentError("Missing env vars: " + ", ".join(_missing))

quick_filter = "ALL"
page_size    = 250

repo_root    = find_repo_root(Path.cwd())
notebook_dir = repo_root / "notebooks" / "used_OSS_by_app"
output_dir   = notebook_dir / "Output"
generated_at = datetime.now(timezone.utc)
date_str     = generated_at.strftime("%Y-%m-%d")
prefix       = section_name.lower().replace(" ", "_")
csv_path     = output_dir / f"{prefix}_used_oss_by_app_{date_str}.csv"
md_path      = output_dir / f"{prefix}_used_oss_by_app_{date_str}.md"

print("Runtime configuration loaded.")
print(f"Section: {section_name}")
print(f"Output directory: {output_dir}")
print(f"CSV output: {csv_path.name}")
print(f"Markdown output: {md_path.name}")


In [ ]:
# Fetch Data from Contrast API
import requests
from collections import defaultdict

def normalize_base_url(raw):
    return raw.rstrip("/") + "/"

def extract_app_id(app):
    return app.get("app_id") or app.get("appID") or app.get("id") or app.get("application_id") or app.get("applicationId") or ""

def extract_app_name(app):
    return app.get("name") or app.get("label") or "UNKNOWN"

def build_headers(auth, api_key):
    return {
        "Authorization": auth,
        "API-Key": api_key,
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

base_url = normalize_base_url(teamserver_url)
headers = build_headers(contrast_auth, contrast_api_key)

# Fetch applications
print("Fetching applications...")
app_map = {}
offset = 0
limit = 100
while True:
    params = {"includeOnlyLicensed": "true", "includeArchived": "false", "offset": offset, "limit": limit}
    resp = requests.get(f"{base_url}Contrast/api/ng/{org_uuid}/applications", headers=headers, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    batch = data.get("applications", [])
    if not batch:
        break
    for app in batch:
        app_id = extract_app_id(app)
        if app_id:
            app_map[app_id] = app.get("name", app_id)
    total = data.get("count", len(batch))
    offset += len(batch)
    print(f"  Applications: {offset}/{total}")
    if offset >= total:
        break

# Fetch servers
print("Fetching servers for environment mapping...")
servers = []
offset = 0
limit = 100
payload = {"quickFilter": "ALL", "tags": [], "applicationsIds": [], "logLevels": [], "serverEnvironments": [], "agentVersions": []}
while True:
    params = {"expand": "applications,assess_protect_status_locked", "sort": "serverName", "offset": offset, "limit": limit}
    resp = requests.post(f"{base_url}Contrast/api/ng/{org_uuid}/servers/filter", headers=headers, params=params, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    batch = data.get("servers", [])
    if not batch:
        break
    servers.extend(batch)
    total = data.get("count", len(batch))
    offset += len(batch)
    print(f"  Servers: {offset}/{total}")
    if offset >= total:
        break

# Build app->environment mapping
app_envs = defaultdict(set)
for srv in servers:
    env = (srv.get("environment") or "UNKNOWN").upper()
    apps = srv.get("applications") or []
    for app in apps:
        app_id = extract_app_id(app)
        if app_id:
            app_envs[app_id].add(env)
app_env_map = {k: sorted(v) for k, v in app_envs.items()}

# Fetch libraries with pagination
print(f"Fetching libraries (paginated, filter={quick_filter})...")
url = f"{base_url}Contrast/api/ng/{org_uuid}/libraries/filter"
all_libs = []
offset = 0
current_limit = page_size
lib_payload = {
    "q": "",
    "quickFilter": quick_filter,
    "apps": [],
    "servers": [],
    "environments": [],
    "grades": [],
    "languages": [],
    "licenses": [],
    "status": [],
    "severities": [],
    "tags": [],
    "includeUnused": False,
    "includeUsed": True,
}

while True:
    params = {"expand": "skip_links,apps,quickFilters,vulns,status,usage_counts", "offset": offset, "limit": current_limit, "sort": "score"}
    try:
        resp = requests.post(url, headers=headers, params=params, json=lib_payload, timeout=90)
        resp.raise_for_status()
    except requests.HTTPError as exc:
        status = exc.response.status_code if exc.response is not None else None
        if status in (400, 413, 422) and current_limit > 50:
            current_limit = 50
            print("  API rejected requested page size; falling back to limit=50")
            continue
        raise
    data = resp.json()
    batch = data.get("libraries", [])
    if not batch:
        break
    all_libs.extend(batch)
    total = data.get("count", len(batch))
    offset += len(batch)
    print(f"  Libraries: {offset}/{total} (limit={current_limit})")
    if offset >= total:
        break

print(f"\n✓ Fetched {len(all_libs)} unique libraries")
print(f"✓ Mapped {len(app_map)} applications")
print(f"✓ Found {len(servers)} servers")

In [ ]:
# Flatten Libraries and Generate Reports
import csv

def _extract_any(lib, *keys, default=""):
    for key in keys:
        if key in lib and lib[key] is not None:
            return lib[key]
    return default

def _extract_apps_for_library(lib):
    apps = lib.get("apps")
    if not isinstance(apps, list) or not apps:
        return [("", "UNSCOPED")]
    pairs = []
    for app in apps:
        if not isinstance(app, dict):
            continue
        app_id = extract_app_id(app)
        app_name = extract_app_name(app)
        pairs.append((app_id, app_name))
    return pairs or [("", "UNSCOPED")]

# Flatten libraries into CSV rows
print("Flattening library data...")
rows = []
for lib in all_libs:
    lib_name = _extract_any(lib, "file_name", "fileName", default="UNKNOWN")
    version = _extract_any(lib, "file_version", "version", default="")
    latest_version = _extract_any(lib, "latest_version", "latestVersion", default="")
    sha1_hash = _extract_any(lib, "sha1", "sha1_hash", "sha1Hash", "hash", "file_hash", "fileHash", default="")
    score = _extract_any(lib, "score", default="")
    grade = str(_extract_any(lib, "grade", default="")).upper()
    classes_used = int(_extract_any(lib, "classes_used", "classesUsed", default=0) or 0)
    class_count = int(_extract_any(lib, "class_count", "totalClasses", default=0) or 0)
    usage = f"{classes_used}/{class_count}" if class_count else str(classes_used)
    cves = int(_extract_any(lib, "total_vulnerabilities", "totalVulnerabilities", default=0) or 0)

    app_pairs = _extract_apps_for_library(lib)
    for app_id, app_name_from_lib in app_pairs:
        app_name = app_map.get(app_id) or app_name_from_lib or "UNSCOPED"
        envs = app_env_map.get(app_id) or ["UNKNOWN"]
        for env in envs:
            rows.append({
                "library_name": lib_name,
                "version": version,
                "latest_version": latest_version,
                "sha1_hash": sha1_hash,
                "cves": cves,
                "usage": usage,
                "application": app_name,
                "application_id": app_id,
                "environment": env,
                "score": score,
                "grade": grade,
                "classes_used": classes_used,
                "total_classes": class_count,
            })

print(f"✓ Flattened into {len(rows)} rows")

# Write CSV
print(f"\nWriting CSV: {csv_path.name}")
output_dir.mkdir(parents=True, exist_ok=True)
fieldnames = [
    "library_name", "version", "latest_version", "sha1_hash", "cves", "usage",
    "application", "application_id", "environment", "score", "grade", "classes_used", "total_classes",
]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print(f"✓ CSV written: {csv_path}")

# Generate Markdown
print(f"\nGenerating Markdown: {md_path.name}")
unique_libs = {(r["library_name"], r["version"]) for r in rows}
unique_apps = {r["application"] for r in rows}

md_lines = []
md_lines.append("# Used OSS by Application and Environment")
md_lines.append("")
md_lines.append(f"**Generated:** {generated_at.strftime('%Y-%m-%d %H:%M:%S')} UTC  ")
md_lines.append(f"**Library quick filter:** {quick_filter}  ")
md_lines.append(f"**Total flattened rows:** {len(rows)}  ")
md_lines.append(f"**Unique libraries:** {len(unique_libs)}  ")
md_lines.append(f"**Applications:** {len(unique_apps)}  ")
md_lines.append("")
md_lines.append("## Summary by Application and Environment")
md_lines.append("")
md_lines.append("| Application | Environment | Libraries | Libraries with CVEs | Total CVEs | Avg Score |")
md_lines.append("|-------------|-------------|-----------|---------------------|------------|-----------|")

group = defaultdict(list)
for row in rows:
    group[(row["application"], row["environment"])].append(row)

for (app, env), entries in sorted(group.items()):
    libs = {(e["library_name"], e["version"]) for e in entries}
    libs_with_cves = {(e["library_name"], e["version"]) for e in entries if int(e.get("cves", 0) or 0) > 0}
    total_cves = sum(int(e.get("cves", 0) or 0) for e in entries)
    scores = [float(e["score"]) for e in entries if str(e.get("score", "")).strip() != ""]
    avg_score = f"{(sum(scores) / len(scores)):.1f}" if scores else "N/A"
    md_lines.append(f"| {app} | {env} | {len(libs)} | {len(libs_with_cves)} | {total_cves} | {avg_score} |")

md_content = "\n".join(md_lines)
output_dir.mkdir(parents=True, exist_ok=True)
with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_content)
print(f"✓ Markdown written: {md_path}")

print(f"\n{'='*60}")
print("✓ REPORT GENERATION COMPLETE")
print(f"{'='*60}")
print(f"CSV:       {csv_path}")
print(f"Markdown:  {md_path}")
print(f"Rows:      {len(rows)}")
print(f"Libraries: {len(all_libs)}")

In [ ]:
# View Results and CLI Command Reference
print("=" * 70)
print("REPORT SUMMARY")
print("=" * 70)
print(f"Generated:   {generated_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"Section:     {section_name}")
print(f"Output Dir:  {output_dir}")
print()
print(f"CSV File:    {csv_path.name}")
print(f"MD File:     {md_path.name}")
print()
print(f"Total Rows:      {len(rows)}")
print(f"Unique Libraries: {len(all_libs)}")
print(f"Applications:    {len(app_map)}")
print()

# Preview first few CSV rows
print("=" * 70)
print("CSV PREVIEW (first 5 rows)")
print("=" * 70)
try:
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(df.head().to_string(index=False))
    print(f"\nTotal CSV rows: {len(df)}")
except ImportError:
    print("(pandas not installed; showing raw CSV format)")
    with open(csv_path) as f:
        for i, line in enumerate(f):
            if i < 6:
                print(line.rstrip())
            else:
                break

# Show MD file location
print("\n" + "=" * 70)
print("MARKDOWN REPORT")
print("=" * 70)
print(f"Location: {md_path}")
print("\nTo view the markdown report, open the file or run:")
print(f"  cat {md_path.relative_to(Path.cwd())}")

# CLI command reference
print("\n" + "=" * 70)
print("EQUIVALENT CLI COMMAND")
print("=" * 70)
print("To run this same report from command line:")
print()
cli_cmd = f"cd {notebook_dir}\n"
cli_cmd += f"python3 ../../Reports/used_OSS_by_app/generate_used_oss_by_app_report.py \\\n"
cli_cmd += f"  --env-file .env \\\n"
cli_cmd += f"  --env-section \"{section_name}\"\n"
if quick_filter != "ALL":
    cli_cmd += f"  --quick-filter \"{quick_filter}\" \\\n"
if page_size != 250:
    cli_cmd += f"  --page-size {page_size}\n"
print(cli_cmd)